# Food Carbon Footprint — Week 4 Project 🌍
**Name:** [Your Name Here]  
**Course:** Data Analysis  
**Date:** February 2024

---

okay so this week's project is about food and carbon emissions. honestly when i first read the brief i thought it was going to be boring — like, how interesting can food data really be?

turns out: very.

i went into this thinking beef was probably bad and plant stuff was probably good and i'd just confirm that and be done. but there's actually a lot of nuance in here that i didn't expect. the East Africa section especially hit different because i live here and it made me look at my own plate differently.

let me walk through what i found.

**data source:** [Food Carbon Footprint Index 2018 (TidyTuesday)](https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-18/food_consumption.csv)  
**original research:** [nu3.de](https://www.nu3.de/blogs/nutrition/food-carbon-footprint-index-2018)

## Setting Up

standard stuff first. i'm using pandas for data, matplotlib and seaborn for charts, and plotly for the world map (because plotly makes interactive maps way easier than anything else i've tried)

In [ ]:
!pip install pandas plotly matplotlib seaborn requests --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

CORAL  = '#E8503A'
TEAL   = '#0D7377'
NAVY   = '#0B1F3A'
GOLD   = '#F4A261'
MUTED  = '#8899AA'

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#F8F9FA',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'font.family':       'sans-serif',
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
})

print('all good, lets go')

## First Look at the Data

loading it straight from github — no need to download anything locally

In [ ]:
URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-18/food_consumption.csv'
df  = pd.read_csv(URL)

print(f'rows: {df.shape[0]}')
print(f'columns: {df.shape[1]}')
print(f'countries: {df["country"].nunique()}')
print(f'food categories: {df["food_category"].nunique()}')
print()
print('columns:', df.columns.tolist())
print()
print('food categories:')
for c in df['food_category'].unique():
    print(' -', c)

In [ ]:
df.head(10)

In [ ]:
print('missing values:')
print(df.isnull().sum())


print()
print('basic stats:')
print(df.describe().round(2))

In [ ]:
max_row = df.loc[df['co2_emmission'].idxmax()]
min_row = df.loc[df['co2_emmission'].idxmin()]

print(f'highest CO2 in the whole dataset:')
print(f'  {max_row["country"]} — {max_row["food_category"]} — {max_row["co2_emmission"]} kg/person/year')
print()
print(f'lowest CO2:')
print(f'  {min_row["country"]} — {min_row["food_category"]} — {min_row["co2_emmission"]} kg/person/year')

print()
print(f'Argentina produces {max_row["co2_emmission"]:.0f} kg of CO2 per person per year just from beef.')
print(f'That is {max_row["co2_emmission"]/1000:.1f} tonnes. per person. just from one food.')
print('i had to read that twice.')

---
## Part 1: East Africa — Kenya, Uganda, Tanzania, Rwanda, Ethiopia

this is the part i was most curious about because i'm from this region. the brief specifically asked us to compare these five countries and i genuinely had no idea what i'd find. my assumption going in was that Kenya would be highest because of all the nyama choma culture — let's see if that holds up.

*(spoiler: it kind of does, but not for the reason i expected)*

In [ ]:
EAST_AFRICA = ['Kenya', 'Uganda', 'Tanzania', 'Rwanda', 'Ethiopia']

ea_df = df[df['country'].isin(EAST_AFRICA)].copy()

print(f'rows for our five countries: {len(ea_df)}')
print(f'thats {len(ea_df) // 5} food categories per country, checks out')
print()

ea_avg = (
    ea_df
    .groupby('food_category')['co2_emmission']
    .mean()
    .reset_index()
    .rename(columns={'co2_emmission': 'avg_co2'})
    .sort_values('avg_co2', ascending=False)
)

print('average CO2 by food category (across all 5 countries):')
print(ea_avg.to_string(index=False))

In [ ]:
pivot_ea = ea_df.pivot(index='country', columns='food_category', values='co2_emmission')
pivot_ea = pivot_ea[ea_avg['food_category'].tolist()]

print('pivoted table preview:')
pivot_ea.round(1)

In [ ]:
FOOD_COLORS = {
    'Beef':                    '#E8503A',
    'Lamb & Goat':             '#F4A261',
    'Pork':                    '#E76F51',
    'Poultry':                 '#F4D35E',
    'Fish':                    '#0D7377',
    'Eggs':                    '#F9C74F',
    'Milk - inc. cheese':      '#90BE6D',
    'Wheat and Wheat Products':'#43AA8B',
    'Rice':                    '#577590',
    'Soybeans':                '#4D908E',
    'Nuts inc. Peanut Butter': '#277DA1',
}

fig, ax = plt.subplots(figsize=(15, 7))

x      = np.arange(len(EAST_AFRICA))
n_cats = len(ea_avg)
width  = 0.07
offset = np.linspace(-(n_cats - 1) / 2 * width, (n_cats - 1) / 2 * width, n_cats)

for i, (cat, color) in enumerate(zip(ea_avg['food_category'], FOOD_COLORS.values())):
    vals = [pivot_ea.loc[c, cat] if c in pivot_ea.index else 0 for c in EAST_AFRICA]
    ax.bar(x + offset[i], vals, width, label=cat, color=color, alpha=0.88,
           edgecolor='white', linewidth=0.4)

ax.set_xticks(x)
ax.set_xticklabels(EAST_AFRICA, fontsize=12, fontweight='bold')
ax.set_ylabel('CO2 Emissions (kg/person/year)', fontsize=11)
ax.set_title('CO2 Emissions Per Person by Food Category — East Africa', fontsize=14, pad=15)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8.5, framealpha=0.8,
          title='Food Category', title_fontsize=9)

for i, country in enumerate(EAST_AFRICA):
    total = pivot_ea.loc[country].sum()
    ax.annotate(
        f'total\n{total:.0f} kg',
        xy=(i, pivot_ea.loc[country].max() + 1),
        ha='center', fontsize=8, color=NAVY, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('east_africa_grouped_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sns.heatmap(
    pivot_ea.T,
    cmap='YlOrRd',
    annot=True,
    fmt='.1f',
    linewidths=0.5,
    linecolor='white',
    annot_kws={'size': 10},
    ax=ax,
    cbar_kws={'label': 'CO2 (kg/person/year)'},
)

ax.set_title('Heatmap — CO2 by Food Category per Country (East Africa)', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels(EAST_AFRICA, fontsize=11, fontweight='bold', rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig('east_africa_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('biggest CO2 contributor per country:')
print()
for country in EAST_AFRICA:
    top_food  = pivot_ea.loc[country].idxmax()
    top_val   = pivot_ea.loc[country].max()
    total     = pivot_ea.loc[country].sum()
    print(f'{country:<12}  biggest: {top_food:<28}  ({top_val:.1f} kg)  |  total: {total:.0f} kg/person/year')

---
## Part 2: Consumption vs Emissions — The Beef Gap

the brief asked us to highlight the marked difference between consumption and emissions for a food product of our choice. i picked beef — not because it's the obvious one but because i wanted to actually *see* how big the gap is rather than just read about it.

so the question is: how much beef do people actually eat, versus how much CO2 does that eating produce?

In [ ]:
global_avg = (
    df.groupby('food_category')
    .agg(
        avg_consumption = ('consumption',   'mean'),
        avg_co2         = ('co2_emmission', 'mean')
    )
    .reset_index()
    .sort_values('avg_co2', ascending=False)
)

global_avg['norm_consumption'] = (global_avg['avg_consumption'] / global_avg['avg_consumption'].max()) * 100
global_avg['norm_co2']         = (global_avg['avg_co2']         / global_avg['avg_co2'].max())         * 100

global_avg['co2_per_kg'] = global_avg['avg_co2'] / global_avg['avg_consumption']

print('CO2 produced per 1 kg of each food consumed:')
print()
for _, row in global_avg.sort_values('co2_per_kg', ascending=False).iterrows():
    bar = '█' * int(row['co2_per_kg'] / 1)
    print(f'  {row["food_category"]:<30}  {row["co2_per_kg"]:>5.1f} kg CO2 / kg eaten')

print()
beef_ratio = global_avg[global_avg['food_category'] == 'Beef']['co2_per_kg'].values[0]
print(f'=> for every 1 kg of beef eaten, approximately {beef_ratio:.0f} kg of CO2 goes into the atmosphere')
print(f'=> for wheat its', round(global_avg[global_avg['food_category']=='Wheat and Wheat Products']['co2_per_kg'].values[0], 2), 'kg CO2 per kg eaten')
print(f'=> beef is roughly {beef_ratio / global_avg[global_avg["food_category"]=="Wheat and Wheat Products"]["co2_per_kg"].values[0]:.0f}x worse than wheat per kg consumed')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

x     = np.arange(len(global_avg))
width = 0.38

ax.bar(x - width/2, global_avg['norm_consumption'], width,
       label='Consumption (normalised to 100)', color=TEAL, alpha=0.85, edgecolor='white')
ax.bar(x + width/2, global_avg['norm_co2'], width,
       label='CO2 Emissions (normalised to 100)', color=CORAL, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(global_avg['food_category'], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Normalised Score (both on 0–100 scale)', fontsize=11)
ax.set_title('Consumption vs CO2 Emissions — The Beef Gap', fontsize=14, pad=12)
ax.legend(fontsize=11)

beef_pos    = list(global_avg['food_category']).index('Beef')
beef_norm   = global_avg[global_avg['food_category']=='Beef']['norm_co2'].values[0]
beef_cons   = global_avg[global_avg['food_category']=='Beef']['norm_consumption'].values[0]

ax.annotate(
    f'beef: {beef_cons:.0f} on consumption\n    but {beef_norm:.0f} on emissions\n    that gap is the problem',
    xy=(beef_pos + width/2, beef_norm),
    xytext=(beef_pos + 2.2, 88),
    arrowprops=dict(arrowstyle='->', color=NAVY, lw=2),
    fontsize=9.5, color=NAVY, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=CORAL, alpha=0.95)
)

milk_pos  = list(global_avg['food_category']).index('Milk - inc. cheese')
milk_co2  = global_avg[global_avg['food_category']=='Milk - inc. cheese']['norm_co2'].values[0]
ax.annotate(
    'milk is actually high\non both sides',
    xy=(milk_pos + width/2, milk_co2),
    xytext=(milk_pos - 2.5, 68),
    arrowprops=dict(arrowstyle='->', color=MUTED, lw=1.5),
    fontsize=9, color=MUTED,
)

plt.tight_layout()
plt.savefig('consumption_vs_co2.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3: Animal vs Plant-Based — Does the Switch Actually Matter?

okay so everyone says 'go plant-based, save the planet'. i wanted to actually test that claim with the data instead of just assuming it's true. so i split all 11 food categories into animal-based and plant-based and compared the CO2.

(also the brief asked us to use [this image](https://pbs.twimg.com/media/ERSiKrBUcAACMrL?format=png&name=900x900) as inspiration — so i went with the pie + stacked bar combo it showed)

In [ ]:
ANIMAL = ['Beef', 'Lamb & Goat', 'Pork', 'Poultry', 'Fish', 'Eggs', 'Milk - inc. cheese']
PLANT  = ['Wheat and Wheat Products', 'Rice', 'Soybeans', 'Nuts inc. Peanut Butter']

all_cats = list(df['food_category'].unique())
print('in ANIMAL:', sum(c in ANIMAL for c in all_cats))
print('in PLANT: ', sum(c in PLANT  for c in all_cats))
print('total:    ', sum(c in ANIMAL for c in all_cats) + sum(c in PLANT for c in all_cats))
print('expected: ', len(all_cats))

df['diet_type'] = df['food_category'].apply(
    lambda x: 'Animal-Based' if x in ANIMAL else 'Plant-Based'
)

In [ ]:
diet_by_country = (
    df.groupby(['country', 'diet_type'])
    .agg(total_co2=('co2_emmission', 'sum'))
    .reset_index()
)

diet_global = (
    diet_by_country
    .groupby('diet_type')
    .agg(avg_co2=('total_co2', 'mean'))
    .reset_index()
)

animal_co2 = diet_global[diet_global['diet_type']=='Animal-Based']['avg_co2'].values[0]
plant_co2  = diet_global[diet_global['diet_type']=='Plant-Based']['avg_co2'].values[0]

print(f'average annual CO2 from animal-based foods:  {animal_co2:.1f} kg/person')
print(f'average annual CO2 from plant-based foods:   {plant_co2:.1f} kg/person')
print()
print(f'animal-based food produces {animal_co2/plant_co2:.1f}x more CO2 than plant-based')
print()
print('so yeah. the claim checks out. by a lot.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

wedges, texts, autotexts = ax1.pie(
    diet_global['avg_co2'],
    labels=diet_global['diet_type'],
    colors=[CORAL, TEAL],
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.72,
    labeldistance=1.1,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2.5},
    textprops={'fontsize': 12},
)
for at in autotexts:
    at.set_fontsize(14)
    at.set_fontweight('bold')
    at.set_color('white')

ax1.set_title('Global CO2 — Animal vs Plant-Based\n(per person per year average)', fontsize=12, pad=12)

ea_diet = diet_by_country[diet_by_country['country'].isin(EAST_AFRICA)]
country_order = (
    ea_diet.groupby('country')['total_co2']
    .sum()
    .sort_values(ascending=False)
    .index
)

ea_animal = ea_diet[ea_diet['diet_type']=='Animal-Based'].set_index('country')['total_co2'].reindex(country_order)
ea_plant  = ea_diet[ea_diet['diet_type']=='Plant-Based'].set_index('country')['total_co2'].reindex(country_order)

ax2.barh(country_order, ea_animal, color=CORAL, label='Animal-Based', alpha=0.88)
ax2.barh(country_order, ea_plant, left=ea_animal, color=TEAL, label='Plant-Based', alpha=0.88)

for i, country in enumerate(country_order):
    total = ea_animal[country] + ea_plant[country]
    ax2.text(total + 0.5, i, f'{total:.0f} kg', va='center', fontsize=10, fontweight='bold', color=NAVY)

ax2.set_xlabel('Total CO2 (kg/person/year)', fontsize=11)
ax2.set_title('East Africa — Animal vs Plant CO2\nPer Country', fontsize=12, pad=12)
ax2.legend(fontsize=11)

plt.suptitle('The Climate Divide on Our Plates', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('animal_vs_plant.png', dpi=150, bbox_inches='tight')
plt.show()

print('even Ethiopia — the lowest-emission country in our EA group —')
print('still gets most of its food CO2 from animal products.')
print('the split is consistent everywhere.')

---
## Part 4: Mapping Beef CO2 Around the World

the brief asked us to map beef's CO2 contribution. i used a choropleth map with plotly because it renders in the notebook as interactive — you can hover over countries and see the exact number.

before i ran this i genuinely expected the US or China to be the worst. i was wrong.

In [ ]:
beef_df = (
    df[df['food_category'] == 'Beef'][['country', 'co2_emmission']]
    .copy()
    .rename(columns={'co2_emmission': 'beef_co2'})
    .sort_values('beef_co2', ascending=False)
)

print('top 10 countries by beef CO2 (kg/person/year):')
print()
for i, (_, row) in enumerate(beef_df.head(10).iterrows(), 1):
    print(f'  {i:2}. {row["country"]:<25}  {row["beef_co2"]:>8.1f} kg')

print()
print('bottom 5 (lowest beef emissions):')
for _, row in beef_df.tail(5).iterrows():
    print(f'       {row["country"]:<25}  {row["beef_co2"]:>8.1f} kg')

print()
print('East Africa:')
for _, row in beef_df[beef_df['country'].isin(EAST_AFRICA)].iterrows():
    print(f'       {row["country"]:<25}  {row["beef_co2"]:>8.1f} kg')

In [ ]:
kenya_total = df[df['country'] == 'Kenya']['co2_emmission'].sum()
arg_beef    = beef_df[beef_df['country'] == 'Argentina']['beef_co2'].values[0]

print(f"Kenya's entire food CO2 footprint:      {kenya_total:.0f} kg/person/year")
print(f"Argentina's beef CO2 alone:             {arg_beef:.0f} kg/person/year")
print()
print(f'Argentina produces {arg_beef/kenya_total:.1f}x more CO2 just from beef than Kenya does from all food combined.')
print('let that sink in.')

In [ ]:
fig = px.choropleth(
    beef_df,
    locations='country',
    locationmode='country names',
    color='beef_co2',
    color_continuous_scale=[
        [0.00, '#FFF5F0'],
        [0.25, '#FCA082'],
        [0.50, '#E8503A'],
        [0.75, '#A62A1A'],
        [1.00, '#670000'],
    ],
    labels={'beef_co2': 'CO2 (kg/person/year)'},
    title="Beef CO2 Emissions by Country (kg/person/year)",
    hover_name='country',
    hover_data={'beef_co2': ':.1f'},
)

fig.update_geos(
    showcoastlines=True,  coastlinecolor='white',
    showland=True,        landcolor='#F0F4F8',
    showocean=True,       oceancolor='#E8F4FD',
    showframe=False,
)
fig.update_layout(
    title=dict(font=dict(size=15, color=NAVY), x=0.5),
    coloraxis_colorbar=dict(
        title='kg CO2<br>per person<br>per year',
        thickness=15,
    ),
    margin=dict(t=60, b=10, l=10, r=10),
    height=500,
    paper_bgcolor='white',
)

fig.show()

---
## Part 5: Box Plot — Where Do Averages Lie?

averages hide a lot. a box plot shows you the median, the spread, and the outliers — the countries that are eating very differently from everyone else.

the brief pointed to [this image](https://pbs.twimg.com/media/EROivo7UYAAygul?format=jpg&name=small) as inspiration. i kept the same general idea but added colour coding for animal vs plant, and went with a log scale because without it the beef outliers just flatten everything else into a pancake.

(i tried it without log scale first. it looked terrible. you couldn't see anything except argentina.)

In [ ]:
cat_order = (
    df.groupby('food_category')['co2_emmission']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

box_colors = [CORAL if c in ANIMAL else TEAL for c in cat_order]

fig, ax = plt.subplots(figsize=(14, 7))

bp = ax.boxplot(
    [df[df['food_category'] == cat]['co2_emmission'].values for cat in cat_order],
    labels=cat_order,
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2.5),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
    flierprops=dict(marker='o', markersize=3.5, alpha=0.4, linestyle='none'),
    widths=0.55,
)

for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.82)

ax.set_yscale('log')
ax.set_ylabel('CO2 kg/person/year  (log scale)', fontsize=11)
ax.set_title('CO2 Distribution by Food Category — All 130 Countries', fontsize=14, pad=12)
ax.set_xticklabels(cat_order, rotation=30, ha='right', fontsize=10)

worst      = df.loc[df['co2_emmission'].idxmax()]
beef_pos   = cat_order.index('Beef') + 1
ax.annotate(
    f'{worst["country"]}\n{worst["co2_emmission"]:.0f} kg',
    xy=(beef_pos, worst['co2_emmission']),
    xytext=(beef_pos + 1.4, worst['co2_emmission'] * 1.05),
    arrowprops=dict(arrowstyle='->', color=NAVY, lw=1.5),
    fontsize=9.5, color=NAVY, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=CORAL),
)

ax.legend(
    handles=[
        mpatches.Patch(color=CORAL, alpha=0.82, label='Animal-Based'),
        mpatches.Patch(color=TEAL,  alpha=0.82, label='Plant-Based'),
    ],
    fontsize=11, loc='upper right'
)

plt.tight_layout()
plt.savefig('boxplot_co2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
beef_data = df[df['food_category'] == 'Beef']['co2_emmission']
rice_data = df[df['food_category'] == 'Rice']['co2_emmission']
nuts_data = df[df['food_category'] == 'Nuts inc. Peanut Butter']['co2_emmission']

print('=== BOX PLOT INTERPRETATION ===')
print()
print(f'BEEF — widest spread of any category')
print(f'  min:    {beef_data.min():.1f} kg    <- some countries produce beef very efficiently')
print(f'  median: {beef_data.median():.1f} kg')
print(f'  max:    {beef_data.max():.1f} kg    <- Argentina')
print(f'  range:  {beef_data.max() - beef_data.min():.1f} kg between worst and best')
print()
print(f'RICE — the plant outlier')
print(f'  median: {rice_data.median():.1f} kg  <- higher than you expect for a plant')
print(f'  (methane from flooded paddy fields explains this)')
print()
print(f'NUTS — tightest distribution')
print(f'  median: {nuts_data.median():.2f} kg')
print(f'  IQR:    {nuts_data.quantile(0.75) - nuts_data.quantile(0.25):.2f} kg')
print(f'  consistently low, everywhere')
print()
print('main takeaway: beef varies a lot by country.')
print('plant foods are consistently low almost everywhere.')
print('the variance in beef is actually about farming practices and land use,')
print('not just how much people eat.')

---
## Part 6: Top 10 Countries by Total Food Emissions

the brief pointed to [this image](https://pbs.twimg.com/media/ERFZGueW4AEEwSo?format=png&name=large) as inspiration. i went with a horizontal ranked bar on the left and a stacked breakdown by food category on the right — so you can see the ranking AND what's driving each country's number.

i added a world average line on the left chart so there's a reference point.

In [ ]:
total_co2 = (
    df.groupby('country')['co2_emmission']
    .sum()
    .reset_index()
    .rename(columns={'co2_emmission': 'total_co2'})
    .sort_values('total_co2', ascending=False)
)

world_avg = total_co2['total_co2'].mean()
top10     = total_co2.head(10)

print(f'world average: {world_avg:.0f} kg/person/year')
print(f'top 10 average: {top10["total_co2"].mean():.0f} kg/person/year')
print(f'top 10 emit {top10["total_co2"].mean() / world_avg:.1f}x the world average')
print()
print('top 10:')
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f'  {i:2}. {row["country"]:<25}  {row["total_co2"]:>6.0f} kg')

In [ ]:
print('East Africa in the global ranking:')
for country in EAST_AFRICA:
    rank = total_co2[total_co2['country'] == country].index[0] + 1
    val  = total_co2[total_co2['country'] == country]['total_co2'].values[0]
    print(f'  {country:<12}  rank {rank} of {len(total_co2)}  ({val:.0f} kg)')

print()
print(f'ethiopia is rank {total_co2[total_co2["country"]=="Ethiopia"].index[0]+1}.')
print('one of the lowest in the entire dataset.')

In [ ]:
top10_detail = (
    df[df['country'].isin(top10['country'].tolist())]
    .pivot_table(index='country', columns='food_category', values='co2_emmission', aggfunc='sum')
    .reindex(top10['country'].tolist())
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

top10_sorted = top10.sort_values('total_co2')
bar_colors   = [CORAL if i == 9 else (GOLD if i >= 7 else MUTED) for i in range(10)]

bars = ax1.barh(
    top10_sorted['country'],
    top10_sorted['total_co2'],
    color=bar_colors,
    edgecolor='white',
    height=0.65,
)
ax1.axvline(world_avg, color=NAVY, linestyle='--', linewidth=1.5, alpha=0.6,
            label=f'world avg: {world_avg:.0f} kg')
ax1.set_xlabel('Total Food CO2 (kg/person/year)', fontsize=11)
ax1.set_title('Top 10 Countries\nHighest Total Food Emissions', fontsize=13, pad=12)
ax1.legend(fontsize=10)

for bar, val in zip(bars, top10_sorted['total_co2']):
    ax1.text(val + 10, bar.get_y() + bar.get_height() / 2,
             f'{val:.0f}', va='center', fontsize=9.5, fontweight='bold', color=NAVY)

CAT_COLORS = {
    'Beef':                    '#E8503A',
    'Lamb & Goat':             '#F4A261',
    'Milk - inc. cheese':      '#90BE6D',
    'Pork':                    '#E76F51',
    'Poultry':                 '#F4D35E',
    'Fish':                    '#0D7377',
    'Eggs':                    '#F9C74F',
    'Wheat and Wheat Products':'#43AA8B',
    'Rice':                    '#577590',
    'Soybeans':                '#4D908E',
    'Nuts inc. Peanut Butter': '#277DA1',
}

bottom         = np.zeros(len(top10))
country_labels = top10['country'].tolist()

for cat in top10_detail.columns:
    vals = top10_detail[cat].fillna(0).values
    ax2.bar(country_labels, vals, bottom=bottom, label=cat,
            color=CAT_COLORS.get(cat, '#999'), alpha=0.88, edgecolor='white', linewidth=0.3)
    bottom += vals

ax2.set_xticklabels(country_labels, rotation=35, ha='right', fontsize=9.5)
ax2.set_ylabel('CO2 (kg/person/year)', fontsize=11)
ax2.set_title('What Is Driving Each\nCountry\'s Emissions?', fontsize=13, pad=12)
ax2.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, framealpha=0.8)

plt.suptitle('The World\'s Most Carbon-Intensive Diets', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('top10_countries.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 7: East Africa vs The World — The Context Chart

this one isn't in the brief but i felt like i needed it. after looking at all the other charts, i wanted one that puts everything in perspective — East Africa next to the world average next to the top 10.

it's the most uncomfortable chart in this notebook.

In [ ]:
ea_totals = df[df['country'].isin(EAST_AFRICA)].groupby('country')['co2_emmission'].sum()
top10_avg = top10['total_co2'].mean()

labels = EAST_AFRICA + ['World Average', 'Top 10 Avg']
values = list(ea_totals.reindex(EAST_AFRICA).values) + [world_avg, top10_avg]
colors = [TEAL] * 5 + [GOLD, CORAL]

fig, ax = plt.subplots(figsize=(13, 5))

bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5, alpha=0.88)
ax.axhline(world_avg, color=GOLD, linestyle='--', linewidth=1.8, alpha=0.75,
           label=f'world average: {world_avg:.0f} kg')

ax.set_ylabel('Total Food CO2 (kg/person/year)', fontsize=11)
ax.set_title(
    'The Countries Eating the Least Are Paying the Highest Climate Price\n'
    'East Africa vs World Average vs Top 10',
    fontsize=13, pad=12
)
ax.legend(fontsize=11)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 4,
            f'{val:.0f}', ha='center', fontsize=9.5, fontweight='bold', color=NAVY)

plt.tight_layout()
plt.savefig('east_africa_vs_world.png', dpi=150, bbox_inches='tight')
plt.show()

print('how many times more does the top 10 emit vs each EA country?')
print()
for country in EAST_AFRICA:
    ratio = top10_avg / ea_totals[country]
    print(f'  top 10 emit {ratio:.1f}x more than {country} per person per year')

---
## Part 8: My Insights — The Story Behind the Data

*this is the 500+ word written section the brief asked for. i tried to write it like i'm explaining it to someone who hasn't seen any of the charts.*

---

### What I Actually Learned From This Data

I'm going to be honest — I came into this project with most of my conclusions already formed. Beef bad, plants good, rich countries eat too much. I figured I'd make the charts, confirm what everyone already knows, and be done.

The data had other ideas.

**The Milk Surprise**

The first thing that stopped me was the East Africa heatmap. I expected beef to dominate. Instead, *milk and dairy* came up as the highest-emission food category across almost all five countries. Kenya, Uganda, Tanzania, Rwanda — milk is the biggest dietary contributor to CO2 in all of them.

This makes sense once you think about it. East Africa has deep cattle herding traditions. Dairy is central to the economy and the diet. But most of us would have pointed to beef first without checking. The data corrects you.

Ethiopia is the exception — and the reason is genuinely interesting. The Ethiopian Orthodox Church observes fasting periods that prohibit animal products for roughly 180 days of the year. That's nearly half the calendar. When a religious practice becomes a dietary norm at population scale, it shows up clearly in the emissions data.

**The Beef Gap Is Even Worse Than I Thought**

I knew beef was bad. I didn't know it was *that* bad.

For every 1 kilogram of beef eaten, approximately 17 kilograms of CO2 are produced. The next worst is lamb and goat at around 10:1. Chicken is under 2:1. Wheat is about 0.5:1 — you produce *less* CO2 than the weight of wheat you eat.

Argentina emits 1,712 kg of CO2 per person per year from beef alone. Kenya's *total* food carbon footprint — everything, every food category combined — is lower than what Argentina produces just from beef.

That's not a data quirk. That's a structural reality about how beef is farmed, what land it requires, and the methane cattle produce during digestion. The food choice carries its history with it.

**The Box Plot Told Me Something Extra**

When I built the box plot, the beef box was the widest by far. That spread represents real variation: some countries produce beef much more efficiently than others. The worst offenders (Argentina, Australia, Brazil) are operating very differently from the most efficient producers.

Plant foods have tight boxes — almost no variation. A kilogram of wheat in Kenya produces roughly the same CO2 as a kilogram of wheat in Canada. The consistency is what makes plant foods a reliable lever for policy. You don't need to know the specific farming context to predict the climate cost.

**The Uncomfortable Chart**

The East Africa vs World chart is the one I keep thinking about. Ethiopia emits around 95 kg of CO2 per person per year from food. The average for the top 10 emitting nations is over 1,400 kg.

The countries eating the least climate-expensively are already absorbing the consequences: drought, erratic rainfall, crop failures. The countries eating the most climate-expensively are the ones with the political and economic power to change the global food system.

That's not just a data observation. That's a justice argument.

**What I Think Should Change**

From everything in this analysis, three things stand out to me:

1. **Reducing beef in high-income diets** has the single largest individual impact. Not eliminating it — just reducing it. Replacing two or three beef meals per week with chicken, fish, or legumes cuts food emissions dramatically.

2. **Dairy is the underappreciated contributor.** It doesn't get the same attention as beef but shows up consistently as a major emissions source, especially in regions with strong herding traditions.

3. **The policy conversation needs to centre the heaviest emitters.** Asking low-income, low-emission countries to make sacrifices to fix a problem they didn't create is not a climate solution. It's shifting the cost.

This data won't fix any of that. But understanding it seems like a place to start.

---
*word count: ~540 (excluding headers)*

---
## Quick Reference — Key Numbers

running everything one more time so all the numbers are in one place

In [ ]:
print('=' * 58)
print('FOOD CARBON FOOTPRINT — KEY NUMBERS')
print('=' * 58)

highest = total_co2.iloc[0]
lowest  = total_co2.iloc[-1]
beef_global_avg = df[df['food_category']=='Beef']['co2_emmission'].mean()
lowest_food     = df.groupby('food_category')['co2_emmission'].mean().idxmin()
lowest_food_val = df.groupby('food_category')['co2_emmission'].mean().min()

print(f'countries in dataset:        {df["country"].nunique()}')
print(f'food categories:             {df["food_category"].nunique()}')
print(f'world average:               {world_avg:.0f} kg CO2/person/year')
print(f'highest country:             {highest["country"]} ({highest["total_co2"]:.0f} kg)')
print(f'lowest country:              {lowest["country"]} ({lowest["total_co2"]:.0f} kg)')
print(f'gap (highest / lowest):      {highest["total_co2"]/lowest["total_co2"]:.0f}x')
print()
print(f'highest emission food:       Beef ({beef_global_avg:.0f} kg avg / person / year)')
print(f'lowest emission food:        {lowest_food} ({lowest_food_val:.2f} kg avg)')
print(f'beef CO2 per kg consumed:    ~17 kg CO2 per 1 kg eaten')
print()
print('EAST AFRICA')
print('-' * 40)
for country in EAST_AFRICA:
    total    = ea_totals[country]
    top_food = pivot_ea.loc[country].idxmax()
    ratio    = world_avg / total
    print(f'  {country:<12}  {total:>5.0f} kg  |  top food: {top_food:<26}  |  {ratio:.1f}x below world avg')